# 11.06 Bedrock Prompt Caching Middleware

AWS Bedrock 경유로 Claude/Nova 같은 모델을 호출할 때 `BedrockPromptCachingMiddleware` 로
**시스템 프롬프트 · 도구 정의 · 마지막 메시지** 위치에 자동으로 캐시 체크포인트를 박아 입력 토큰 비용을 낮춘다.

`AnthropicPromptCachingMiddleware`(11.01)와 **파라미터 이름과 의미는 거의 동일**하지만 대상 패키지와 모델별 제약이 다르다.

## 학습 목표

- `BedrockPromptCachingMiddleware` 의 4개 파라미터(`type`, `ttl`, `min_messages_to_cache`, `unsupported_model_behavior`) 를 이해한다
- `usage_metadata.input_token_details` 에서 `cache_creation` / `cache_read` 를 읽어 캐시 적중을 검증한다
- `ChatBedrock` vs `ChatBedrockConverse` 에서 `type` 파라미터 처리가 어떻게 다른지 안다
- Nova 모델의 제약(5m TTL 전용, tool 정의 캐시 미지원) 을 구분한다

## 언제 쓰나

- AWS 계정·VPC 경계 안에서만 LLM 을 호출해야 하는 기업 환경 (Bedrock 경유)
- Claude 를 Anthropic 직접 API 가 아닌 **Bedrock 요금제**로 이용할 때
- Amazon Nova 모델을 쓰면서 긴 시스템 프롬프트를 반복 사용할 때

## 11.06.1 환경 설정

필요 패키지: `langchain`, `langchain-aws`. AWS 자격증명(`AWS_ACCESS_KEY_ID` 등)과 리전이 `.env` 또는 환경에 설정돼 있어야 한다.
모델별로 Bedrock 콘솔에서 **모델 액세스**를 먼저 허용해야 호출이 된다.

In [ ]:
# !pip install -U langchain langchain-aws

from dotenv import load_dotenv
load_dotenv(override=True)

import os
print("AWS_REGION:", os.getenv("AWS_REGION") or os.getenv("AWS_DEFAULT_REGION"))

## 11.06.2 ChatBedrockConverse + Claude 조합 (권장)

Converse API 는 Bedrock의 **통합 인터페이스**로, 다양한 모델을 같은 API 로 호출할 수 있다.
`BedrockPromptCachingMiddleware` 를 붙이면 system/tool/last-message 위치에 자동으로 캐시 체크포인트가 들어간다.

> 주의: **체크포인트가 실제로 작동하려면 캐시 대상 블록이 ~1,024 토큰 이상** 이어야 한다.
> 짧은 system prompt 에서는 `cache_creation` 이 잡히지 않을 수 있다.

In [ ]:
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_aws import ChatBedrockConverse
from langchain_aws.middleware.prompt_caching import BedrockPromptCachingMiddleware

@tool
def get_region(zone: str) -> str:
    """zone 에 해당하는 AWS 리전을 반환."""
    return f"{zone} -> us-west-2"

LONG_SYSTEM = (
    "당신은 사내 클라우드 비용 최적화 에이전트입니다. 아래 정책을 항상 준수합니다.\n\n"
    + ("\n".join(f"- 정책 {i}: 비용 관련 조회 시 리전과 시점을 명시한다." for i in range(1, 80)))
)

agent_bedrock = create_agent(
    model=ChatBedrockConverse(
        model="us.anthropic.claude-sonnet-4-5-20250929-v1:0",
        max_tokens=1024,
    ),
    tools=[get_region],
    system_prompt=LONG_SYSTEM,
    middleware=[BedrockPromptCachingMiddleware(ttl="5m")],
)

## 11.06.3 캐시 적중 검증

두 번 연속 호출하고 `usage_metadata.input_token_details` 를 읽어 캐시 생성/적중을 확인한다.
`cache_creation` 은 1회차에서 커지고, `cache_read` 는 2회차 이후에 채워진다.

In [ ]:
def show_usage(result):
    last = result["messages"][-1]
    um = getattr(last, "usage_metadata", None) or {}
    print("input/output:", um.get("input_tokens"), "/", um.get("output_tokens"))
    details = um.get("input_token_details", {})
    print("cache_creation:", details.get("cache_creation"))
    print("cache_read    :", details.get("cache_read"))

r1 = agent_bedrock.invoke({"messages": [{"role": "user", "content": "us-east-1 리전 소개"}]})
print("--- 1회차 ---"); show_usage(r1)

r2 = agent_bedrock.invoke({"messages": [{"role": "user", "content": "ap-northeast-2 리전 소개"}]})
print("\n--- 2회차 ---"); show_usage(r2)

## 11.06.4 파라미터 전체

| 파라미터 | 기본값 | 설명 |
|---------|--------|------|
| `type` | `"ephemeral"` | ChatBedrock 전용. ChatBedrockConverse 는 이 값을 무시하고 `"default"` 를 쓴다 |
| `ttl` | `"5m"` | `"5m"` 또는 `"1h"`. **Nova 계열은 `"5m"` 만 지원** |
| `min_messages_to_cache` | `0` | 메시지 수가 이 이상일 때만 cache 체크포인트 부착 |
| `unsupported_model_behavior` | `"warn"` | 캐시 미지원 모델에 대해 `"ignore"` / `"warn"` / `"raise"` |

## 11.06.5 ChatBedrock (Invoke 모델) 예시

`ChatBedrock` 은 이전 세대 invoke-model 래퍼다. `BedrockPromptCachingMiddleware` 는 이쪽도 지원하며
`type="ephemeral"` 파라미터가 **여기서는 실제로 반영**된다.

ChatBedrock 쪽은 Anthropic 모델 한정으로 **tool 정의 캐시 + 확장 TTL(1h)** 를 모두 지원한다.

In [ ]:
from langchain_aws import ChatBedrock

agent_bedrock_invoke = create_agent(
    model=ChatBedrock(
        model_id="anthropic.claude-3-5-sonnet-20241022-v2:0",
        model_kwargs={"max_tokens": 1024},
    ),
    tools=[get_region],
    system_prompt=LONG_SYSTEM,
    middleware=[
        BedrockPromptCachingMiddleware(
            type="ephemeral",
            ttl="1h",                     # Anthropic on ChatBedrock 은 1h 가능
            min_messages_to_cache=2,
            unsupported_model_behavior="warn",
        ),
    ],
)
print("agent_bedrock_invoke 준비 완료")

## 11.06.6 Amazon Nova 제약

| 항목 | ChatBedrockConverse + Anthropic | ChatBedrockConverse + Nova |
|------|-------------------------------|---------------------------|
| 시스템 프롬프트 캐시 | O | O |
| 도구 정의 캐시 | O | **X** |
| 메시지 캐시 | O | O (tool result 메시지는 제외) |
| TTL `1h` | O | **X (5m 만)** |

Nova 모델을 대상으로 설정할 때는 `ttl="5m"` 로 고정하고, `unsupported_model_behavior="warn"` 으로
실수로 1h 를 지정해도 조용히 무시되지 않도록 경고를 남긴다.

## 11.06.7 Anthropic 미들웨어와 동시에 쓰지 말 것

`AnthropicPromptCachingMiddleware` 와 `BedrockPromptCachingMiddleware` 는 **둘 다 `cache_control` 마커**를 삽입한다.
**같은 에이전트에 둘 다 붙이면** 마커가 중복되어 에러 또는 예기치 않은 캐시 동작이 날 수 있다.
모델 종류로 **택일**한다.

## 체크리스트

- [ ] 캐시 대상 블록이 ~1,024 토큰 이상인지 확인 (짧으면 캐시가 안 붙는다)
- [ ] Converse 사용 시 `type` 값이 무시된다는 점을 문서화했는가
- [ ] Nova 타깃이면 `ttl="5m"` 로 고정했는가
- [ ] Anthropic 직접 호출과 Bedrock 호출을 **라우팅으로 분리**했는가

## 다음 노트북

- `07_openai_moderation.ipynb` — OpenAI Moderation 으로 입출력 안전 필터링

## 참고

- 공식: https://docs.langchain.com/oss/python/integrations/middleware/aws
- AWS Bedrock Prompt Caching: https://docs.aws.amazon.com/bedrock/latest/userguide/prompt-caching.html